In [ ]:
import os

import matplotlib.pyplot as plt
import scipy.sparse as sp
import scanpy as sc
import harmonypy as hm
import lamindb as ln

ln.track("RcgUzefpkneN")

In [ ]:
SWEEP_DIR = '/home/workspace/PCD_data/xenium/commercial/data/resolution_sweep_v2'

os.makedirs(SWEEP_DIR, exist_ok=True)

In [ ]:
ln.Artifact.filter(key__contains='h5ad').df()
# ln.Artifact.filter(key="test_combined_filtered.h5ad").df()

In [ ]:
combined = ln.Artifact.get(uid='FGSKxkur6ovhRglZ0000').load()

In [ ]:
n_pca_comps = 30
n_neighbors = 15

sc.pp.normalize_total(combined, inplace=True)
sc.pp.log1p(combined)
combined.raw = combined
sc.pp.scale(combined, zero_center=False)
sc.pp.pca(combined, n_comps=n_pca_comps, random_state=0)

harmony = hm.run_harmony(combined.obsm['X_pca'], combined.obs, vars_use='sample', random_state=0)

harmony_pca = harmony.Z_corr
if harmony_pca.shape[0] != combined.n_obs:
    harmony_pca = harmony_pca.T
combined.obsm['X_pca_harmony'] = harmony_pca

sc.pp.neighbors(combined, use_rep='X_pca_harmony', n_neighbors=n_neighbors, n_pcs=n_pca_comps, random_state=0)
sc.tl.umap(combined, random_state=0, spread=2)

In [ ]:
processing_label  = ln.ULabel(name='processing').save()
clustered_label   = ln.ULabel(name='clustered').save()

In [ ]:
# DEG params
ln.Param(name='resolution',     dtype=float).save()
ln.Param(name='deg_method',     dtype=str).save()
ln.Param(name='clustering_key', dtype=str).save()
ln.Param(name='n_clusters',     dtype=int).save()

# Adata params
ln.Param(name='harmony',        dtype=bool).save()
ln.Param(name='n_neighbors',    dtype=int).save()
ln.Param(name='n_pcs',          dtype=int).save()
ln.Param(name='spread',         dtype=float).save()

In [ ]:
RESOLUTIONS = [round(0.1 * i, 2) for i in range(1, 12)]  # 0.1, 0.2, …, 1.1
sweep_keys = [f'leiden_res{res:.2f}' for res in RESOLUTIONS]
for res, key in zip(RESOLUTIONS, sweep_keys):
    sc.tl.leiden(combined, flavor='igraph', n_iterations=2, directed=False,
                 resolution=res, key_added=key, random_state=0)

    sc.tl.rank_genes_groups(combined, key, method='wilcoxon', use_raw=True, key_added=f'rank_{key}')
    deg = sc.get.rank_genes_groups_df(combined, group=None, key=f'rank_{key}')
    deg_stem = f'commercial_deg_{key}_wilcoxon_v2.csv'
    deg_path = os.path.join(SWEEP_DIR, deg_stem)
    deg.to_csv(deg_path, index=False)
    
    artifact = ln.Artifact(deg_path, key=deg_stem,
                               description=f'Commercial DEG markers, leiden res {res:.2f}, Wilcoxon, log1p raw').save()
    artifact.features.add_values({
        'resolution':     res,
        'deg_method':     'wilcoxon',
        'clustering_key': key,
        'n_clusters':     combined.obs[key].nunique()
    })

In [ ]:
# Writes a new lamindb artifact (key 'commercial_combined_clustered.h5ad'); the QC-filtered input artifact is not touched.
artifact = ln.Artifact.from_anndata(combined, key='commercial_combined_clustered.h5ad',
                                    description='Combined commercial Xenium object after batch correction, clustering, and resolution sweep').save()
artifact.ulabels.add(processing_label, clustered_label)
artifact.features.add_values({'harmony': True, 'n_neighbors': n_neighbors, 'n_pcs': n_pca_comps, 'spread': 2.0})

ln.finish()